In [1]:
import glob
import torch
import os
import json
from pathlib import Path
from PIL import Image
import random
from datasets import Dataset
from transformers import DonutProcessor, VisionEncoderDecoderModel
from transformers import VisionEncoderDecoderConfig
from transformers import GenerationConfig

from typing import Any, List, Tuple
from datasets.info import DatasetInfo
from datasets.splits import NamedSplit
from datasets.table import Table


import lightning as L
from torch.utils.data import DataLoader

from pathlib import Path
import re
from nltk import edit_distance
import numpy as np
import math

from torch.nn.utils.rnn import pad_sequence
from torch.optim.lr_scheduler import LambdaLR


import mlflow

In [2]:
image_height=560
image_width=560


# the donut model uses CrossEntropyLoss as loss function for training
# we need to give the ignore-index for pad_token, let CrossEntropyLoss function ignore the loss values of this part
loss_ignore_index=-100

In [3]:
# !pip install -U transformers

In [5]:
config = VisionEncoderDecoderConfig.from_pretrained("naver-clova-ix/donut-base")
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")


Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extractor configuration. Please open a PR/issue to update `preprocessor_config.json` to use `image_processor_type` instead of `feature_extractor_type`. This warning will be removed in v4.40.


In [6]:
processor.tokenizer

XLMRobertaTokenizerFast(name_or_path='naver-clova-ix/donut-base', vocab_size=57522, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'sep_token': '</s>', 'pad_token': '<pad>', 'cls_token': '<s>', 'mask_token': '<mask>', 'additional_special_tokens': ['<s_iitcdip>', '<s_synthdog>']}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	57521: AddedToken("<mask>", rstrip=False, lstrip=True, single_word=False, normalized=True, special=Tru

In [7]:
annotations_dir="/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar_annotations/"
annotations_files=glob.glob(annotations_dir+"*")
print(annotations_files[0])
print(len(annotations_files))

image_dir="/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar/"
images_files=glob.glob(image_dir+"*")
print(images_files[0])
print(len(images_files))

/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar_annotations/75c0449f6917.json
19189
/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar/b2ab3b743d4e.jpg
19189


In [9]:
## some extra special tokens
title_start_token="<title>"
title_end_token="</title>"
x_axis_title_start_token="<x_axis_title>"
x_axis_title_end_token="</x_axis_title>"
y_axis_title_start_token="<y_axis_title>"
y_axis_title_end_token="</y_axis_title>"
x_ticks_start_token="<x_ticks>"
x_ticks_end_token="</x_ticks>"
y_ticks_start_token="<y_ticks>"
y_ticks_end_token="</y_ticks>"
data_series_start_token="<data_series>"
data_series_end_token="</data_series>"


special_tokens = [
    "<title>", "</title>",
    "<x_axis_title>", "</x_axis_title>",
    "<y_axis_title>", "</y_axis_title>",
    "<x_ticks>", "</x_ticks>",
    "<y_ticks>", "</y_ticks>",
    "<data_series>", "</data_series>"
]

In [11]:
# setup the config file
config.encoder.image_size = (image_height, image_width)
# config.decoder.max_length = max_length
config.decoder.max_length = 286
# add necessary tokens into config
config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s>")
config.pad_token_id = processor.tokenizer.convert_tokens_to_ids("<pad>")
config.eos_token_id = processor.tokenizer.convert_tokens_to_ids("</s>")

In [12]:
# setup the processor

# update image size
processor.image_processor.size = {
    "height": image_height,
    "width": image_width,
}

# add special tokens into processor
processor.tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})

12

In [16]:
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", config=config)


In [18]:
# extend decoder embedding layer
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.eos_token_id = processor.tokenizer.eos_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s>")
# model.config.max_length =max_length
model.config.max_length =286
model.config.pad_token_id=processor.tokenizer.pad_token_id

model.config.encoder.pad_token_id=processor.tokenizer.pad_token_id
model.config.decoder.bos_token_id=processor.tokenizer.bos_token_id
model.config.encoder.bos_token_id=processor.tokenizer.bos_token_id
model.config.update({"special_tokens": processor.tokenizer.additional_special_tokens})

In [19]:
print("decoder_start_token_id:", model.config.decoder_start_token_id)
print("token:", processor.tokenizer.convert_ids_to_tokens(model.config.decoder_start_token_id))
print("model.config.max_length:", model.config.max_length)
print("vocab size:", len(processor.tokenizer))
print(model.config.pad_token_id)
print(processor.tokenizer.convert_tokens_to_ids("<title>"))
print(model.decoder.get_input_embeddings().weight.shape)  

decoder_start_token_id: 0
token: <s>
model.config.max_length: 286
vocab size: 57537
1
57525
torch.Size([57537, 1024])


In [20]:
from torch.optim.lr_scheduler import LambdaLR
import math

def get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps, num_cycles=0.5, last_epoch=-1):
    def lr_lambda(current_step):
        if current_step < warmup_steps:
            return float(current_step) / float(max(1, warmup_steps))
        progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * num_cycles * 2.0 * progress)))
    return LambdaLR(optimizer, lr_lambda, last_epoch=last_epoch)

In [21]:
class DonutModelPLModule(L.LightningModule):
    def __init__(self, config, processor, model):
        super().__init__()
        self.config = config
        self.processor = processor
        self.model = model

        if self.config.get("freeze_encoder", True):
            self.freeze_encoder()

        self.print_trainable_parameters()

        self.generation_cfg = GenerationConfig(
            max_length=self.config.get("max_length", 768),
            early_stopping=True,
            do_sample=False,
            num_beams=4,
            use_cache=True,
            bad_words_ids=[[self.processor.tokenizer.unk_token_id]],
            pad_token_id=self.processor.tokenizer.pad_token_id,
            eos_token_id=self.processor.tokenizer.eos_token_id,
            return_dict_in_generate=True,
        )

    def freeze_encoder(self):
        for param in self.model.encoder.parameters():
            param.requires_grad = False

    def print_trainable_parameters(self):
        trainable_params = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total_params = sum(p.numel() for p in self.parameters())
        print(f"Trainable parameters: {trainable_params} / {total_params}")

    def training_step(self, batch, batch_idx):
        pixel_values, labels, _ = batch
        pixel_values = pixel_values.to(self.device)
        labels = labels.to(self.device)

        outputs = self.model(pixel_values, labels=labels)
        loss = outputs.loss

        # mlflow.log_metric("Training Batch Loss", loss.item(), step=batch_idx)
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx, dataset_idx=0):
        pixel_values, labels, answers = batch
        pixel_values = pixel_values.to(self.device)
        labels = labels.to(self.device)
    
        # CrossEntropyLoss (token-level)
        with torch.no_grad():
            outputs = self.model(pixel_values, labels=labels)
            val_loss = outputs.loss
    
            # Log to MLflow & Lightning both
            # mlflow.log_metric("val_loss", val_loss.item(), step=batch_idx)
            self.log("val_loss", val_loss)
    
        # Optional: Edit Distance evaluation
        if self.config.get("log_edit_distance", False):
            decoder_input_ids = torch.full(
                (pixel_values.size(0), 1),
                self.model.config.decoder_start_token_id,
                device=self.device,
            )
    
            generated = self.model.generate(
                pixel_values,
                decoder_input_ids=decoder_input_ids,
                generation_config=self.generation_cfg,
            )
    
            predictions = self.processor.tokenizer.batch_decode(
                generated.sequences, skip_special_tokens=True
            )
    
            scores = []
            for pred, ans in zip(predictions, answers):
                denom = max(len(pred), len(ans))
                score = edit_distance(pred, ans) / denom if denom > 0 else 1.0
                scores.append(score)
    
            # 📝 Log Edit Distance
            avg_score = np.mean(scores)
            # mlflow.log_metric("val_edit_distance", avg_score, step=batch_idx)
            self.log("val_edit_distance", avg_score)
    
        return {"val_loss": val_loss}


    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        pixel_values = batch.to(self.device)
        batch_size = pixel_values.shape[0]

        decoder_input_ids = torch.full(
            (batch_size, 1),
            self.model.config.decoder_start_token_id,
            dtype=torch.long,
            device=self.device,
        )

        outputs = self.model.generate(
            pixel_values,
            decoder_input_ids=decoder_input_ids,
            generation_config=self.generation_cfg,
        )

        predictions = self.processor.tokenizer.batch_decode(
            outputs.sequences, skip_special_tokens=True
        )

        return predictions

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.config.get("lr", 5e-5))
    
        total_steps = self.config.get("max_epochs", 30) * self.config.get("steps_per_epoch", 1000)
        warmup_steps = self.config.get("warmup_steps", int(0.1 * total_steps))
    
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            warmup_steps=warmup_steps,
            total_steps=total_steps,
            num_cycles=0.5
        )
    
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "step",  # update every step
                "frequency": 1,
            }
        }

In [26]:
train_config = {"max_epochs":30,
          "val_check_interval":1, # how many times we want to validate during an epoch
          "check_val_every_n_epoch":1,
          "gradient_clip_val":0.1,
          "lr":1e-6,
          "train_batch_sizes": [32],
          "val_batch_sizes": [32],
          "num_nodes": 4,
          # "warmup_steps": 300,
          "result_path": "./donut/result",
          "log_edit_distance": False,
          "verbose": True,
          }

In [23]:
model_module = DonutModelPLModule(train_config, processor, model)

Trainable parameters: 127683584 / 201864312


In [46]:
model.config

VisionEncoderDecoderConfig {
  "_name_or_path": "naver-clova-ix/donut-base",
  "architectures": [
    "VisionEncoderDecoderModel"
  ],
  "decoder": {
    "_name_or_path": "",
    "activation_dropout": 0.0,
    "activation_function": "gelu",
    "add_cross_attention": true,
    "add_final_layer_norm": true,
    "architectures": null,
    "attention_dropout": 0.0,
    "bad_words_ids": null,
    "begin_suppress_tokens": null,
    "bos_token_id": 0,
    "chunk_size_feed_forward": 0,
    "classifier_dropout": 0.0,
    "cross_attention_hidden_size": null,
    "d_model": 1024,
    "decoder_attention_heads": 16,
    "decoder_ffn_dim": 4096,
    "decoder_layerdrop": 0.0,
    "decoder_layers": 4,
    "decoder_start_token_id": null,
    "diversity_penalty": 0.0,
    "do_sample": false,
    "dropout": 0.1,
    "early_stopping": false,
    "encoder_attention_heads": 16,
    "encoder_ffn_dim": 4096,
    "encoder_layerdrop": 0.0,
    "encoder_layers": 12,
    "encoder_no_repeat_ngram_size": 0,
    "e

In [27]:
from lightning.pytorch import Trainer

ckpt_path = "/Users/yiding/personal_projects/ML/github_repo/donut/trained_models/Donut-epoch=29-val_loss=1.35.ckpt"
trainer = Trainer()
model_module = DonutModelPLModule.load_from_checkpoint(ckpt_path, config=train_config, processor=processor, model=model)

GPU available: True (mps), used: True


TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


Trainable parameters: 127683584 / 201864312


In [ ]:
from PIL import Image

# load image
image = Image.open("/Users/yiding/personal_projects/ML/github_repo/donut/data/image_classes/vertical_bar/348cdffe48e6.jpg").convert("RGB")

# preprocess
pixel_values = processor(image, return_tensors="pt").pixel_values.to("cpu")

# inference
model_module.eval()
with torch.no_grad():
    outputs = model_module.model.generate(
        pixel_values,
        decoder_input_ids=torch.full((1, 1), model.config.decoder_start_token_id, device="cpu"),
        # max_length=model.config.decoder.max_length,
        max_length=512,
        num_beams=4
    )

# 解码
result = processor.tokenizer.decode(outputs[0], skip_special_tokens=False)
print("🔍 Inference Result:", result)

🔍 Inference Result: <s> male ratio bir for 1950<x_axis_title> sextio</title> Country<y_axis_title> sextio<y_axis_title> Maleici sextio</y_axis_title> Newed|th|th|th|thdaleGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusGusEltEltEltEltEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletEletElet Elchie EstiElet El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Gui El Glu Glu Glu Glu Glu Glu Glu Glu Glu Glu Glu Gl

In [ ]:
# save processer, config, model

save_path="/Users/yiding/personal_projects/ML/github_repo/donut/src/test/content_recognition/donut-artifacts"

processor.save_pretrained(save_path)
model.config.save_pretrained(save_path)
model.save_pretrained(save_path)

Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 286}
Some non-default generation parameters are set in the model config. These should go into a GenerationConfig file (https://huggingface.co/docs/transformers/generation_strategies#save-a-custom-decoding-strategy-with-your-model) instead. This warning will be raised to an exception in v4.41.
Non-default generation parameters: {'max_length': 286}
Removed shared tensor {'decoder.lm_head.weight'} while saving. This should be OK, but check by verifying that you don't receive any warning while reloading
